# Load HDX (Humanitarian Data Exchange) Dataset Metadata

This notebook fetches humanitarian indicator datasets from HDX - covering crisis response, food security, displacement, and development data.

In [ ]:
%pip install requests tqdm --quiet

In [ ]:
# Configuration
CATALOG = "main_catalog"
SCHEMA = "dev"
TABLE_NAME = "hdx_indicators"
FULL_TABLE_NAME = f"{CATALOG}.{SCHEMA}.{TABLE_NAME}"

# Search terms for humanitarian indicators
SEARCH_TERMS = [
    "food prices",
    "food security",
    "displacement",
    "refugees",
    "population",
    "health indicators",
    "poverty",
    "education",
    "water sanitation",
    "humanitarian needs",
]

MAX_PER_SEARCH = 500
FRESH_START = False

In [ ]:
import requests
from tqdm import tqdm
from pyspark.sql.types import StructType, StructField, StringType

In [ ]:
schema = StructType([
    StructField("indicator_id", StringType(), False),
    StructField("indicator_name", StringType(), True),
    StructField("long_definition", StringType(), True),
    StructField("source_organization", StringType(), True),
    StructField("source", StringType(), True),
    StructField("topics", StringType(), True),
    StructField("unit", StringType(), True),
    StructField("periodicity", StringType(), True),
    StructField("aggregation_method", StringType(), True),
    StructField("license_type", StringType(), True),
    StructField("embedding_text", StringType(), True),
])

HDX_BASE = "https://data.humdata.org/api/3/action/package_search"

In [ ]:
# Get existing indicator IDs or create table
existing_ids = set()

if FRESH_START:
    spark.sql(f"DROP TABLE IF EXISTS {FULL_TABLE_NAME}")
    print("Fresh start - dropped existing table")
else:
    try:
        existing_df = spark.sql(f"SELECT indicator_id FROM {FULL_TABLE_NAME}")
        existing_ids = set(row.indicator_id for row in existing_df.collect())
        print(f"Resuming - found {len(existing_ids)} existing records")
    except:
        print("Table doesn't exist yet, starting fresh")

if not existing_ids or FRESH_START:
    empty_df = spark.createDataFrame([], schema)
    empty_df.write \
        .format("delta") \
        .option("delta.enableChangeDataFeed", "true") \
        .mode("overwrite") \
        .saveAsTable(FULL_TABLE_NAME)
    
    # Set table description
    spark.sql(f"""
        COMMENT ON TABLE {FULL_TABLE_NAME} IS 
        'Humanitarian Data Exchange dataset metadata covering crisis response, food security, displacement, and development data.'
    """)
    print(f"Created table {FULL_TABLE_NAME} with CDF enabled")

In [ ]:
def fetch_hdx_datasets(query: str, rows: int = 100, start: int = 0):
    """Fetch datasets from HDX."""
    params = {
        "q": query,
        "rows": rows,
        "start": start,
    }
    resp = requests.get(HDX_BASE, params=params, timeout=60)
    resp.raise_for_status()
    return resp.json().get("result", {}).get("results", [])

In [ ]:
# Fetch and load datasets for each search term
for search_term in tqdm(SEARCH_TERMS, desc="Processing search terms"):
    try:
        print(f"\nSearching for: {search_term}")
        
        start = 0
        total_loaded = 0
        
        while start < MAX_PER_SEARCH:
            datasets = fetch_hdx_datasets(search_term, rows=100, start=start)
            if not datasets:
                break
            
            for ds in tqdm(datasets, desc=f"Loading '{search_term}'", leave=False):
                try:
                    dataset_id = ds.get("id", "")
                    
                    if dataset_id in existing_ids:
                        continue
                    
                    title = ds.get("title", "") or ""
                    notes = ds.get("notes", "") or ""
                    org = ds.get("organization", {}) or {}
                    org_title = org.get("title", "") if isinstance(org, dict) else ""
                    
                    # Extract tags/topics
                    tags = ds.get("tags", []) or []
                    topics = ", ".join([t.get("display_name", "") for t in tags[:5] if isinstance(t, dict)])
                    
                    # HDX has location info
                    locations = ds.get("groups", []) or []
                    location_names = ", ".join([loc.get("title", "") for loc in locations[:3] if isinstance(loc, dict)])
                    if location_names:
                        topics = f"{topics}; Locations: {location_names}" if topics else f"Locations: {location_names}"
                    
                    license_title = ds.get("license_title", "") or ""
                    
                    embedding_text = f"{title}. {notes}".strip()
                    if not embedding_text or embedding_text == ".":
                        embedding_text = title or dataset_id

                    record = [(
                        dataset_id,
                        title,
                        notes[:5000] if notes else "",
                        org_title,
                        "Humanitarian Data Exchange (HDX)",
                        topics,
                        "",
                        "",
                        "",
                        license_title,
                        embedding_text[:5000] if len(embedding_text) > 5000 else embedding_text,
                    )]

                    row_df = spark.createDataFrame(record, schema)
                    row_df.write.format("delta").mode("append").saveAsTable(FULL_TABLE_NAME)
                    existing_ids.add(dataset_id)
                    total_loaded += 1
                    
                except Exception as e:
                    print(f"Error loading {dataset_id}: {e}")
                    continue
            
            start += 100
        
        print(f"Loaded {total_loaded} datasets for '{search_term}'")
                
    except Exception as e:
        print(f"Error searching '{search_term}': {e}")
        continue

print("\nDone!")

In [ ]:
# Verify
count = spark.sql(f"SELECT COUNT(*) FROM {FULL_TABLE_NAME}").collect()[0][0]
print(f"Total HDX datasets in table: {count}")
display(spark.sql(f"SELECT * FROM {FULL_TABLE_NAME} LIMIT 5"))